In [1]:


%pip install openai langchain langchain-openai requests -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\lenov\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import requests
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from dotenv import load_dotenv
from langgraph.prebuilt import create_react_agent

load_dotenv()

if not os.getenv("OPENAI_BASE_URL"):
    raise ValueError("Falta OPENAI_BASE_URL en .env")
if not os.getenv("GITHUB_TOKEN"):
    raise ValueError("Falta GITHUB_TOKEN en .env")

stream_handler = StreamingStdOutCallbackHandler()

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    streaming=True,
    callbacks=[stream_handler],
    request_timeout=600,
    temperature=0
)

print("✓ Modelo configurado con streaming habilitado")
print(f"Modelo: {llm.model_name}")
print(f"Streaming: {llm.streaming}")

✓ Modelo configurado con streaming habilitado
Modelo: gpt-4o
Streaming: True


In [3]:


CENTROS = {
    "ensenada": {"lat": -41.140459, "lon": -72.404236, "nombre": "Piscicultura Petrohué"},
    "puelche":  {"lat": -41.733,    "lon": -73.602,    "nombre": "Centro Puelche"},
    "huito":    {"lat": -41.783,    "lon": -73.583,    "nombre": "Centro Huito (San José)"}
}

@tool
def get_clima_actual(centro: str) -> str:
    """Obtiene el clima actual para un centro de cultivo de Camanchaca.
    El parámetro centro puede ser: ensenada, puelche o huito."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado. Opciones: ensenada, puelche, huito."
    
    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&current=temperature_2m,wind_speed_10m,precipitation,weathercode"
        f"&timezone=America/Santiago"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        current = data["current"]
        
        temp   = current["temperature_2m"]
        viento = current["wind_speed_10m"]
        lluvia = current["precipitation"]
        codigo = current["weathercode"]
        
        condicion = "Despejado" if codigo < 3 else "Nublado" if codigo < 50 else "Lluvia"
        
        return (
            f"Centro: {datos['nombre']}\n"
            f"Temperatura: {temp}°C\n"
            f"Viento: {viento} km/h\n"
            f"Precipitación: {lluvia} mm\n"
            f"Condición: {condicion}"
        )
    except Exception as e:
        return f"Error al obtener datos climáticos: {e}"


@tool
def get_pronostico_semana(centro: str) -> str:
    """Obtiene el pronóstico climático de 7 días para un centro de cultivo de Camanchaca.
    El parámetro centro puede ser: ensenada, puelche o huito."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado. Opciones: ensenada, puelche, huito."
    
    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,weathercode"
        f"&timezone=America/Santiago"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        daily = data["daily"]
        
        resultado = f"Pronóstico 7 días - {datos['nombre']}:\n"
        for i in range(7):
            fecha   = daily["time"][i]
            tmax    = daily["temperature_2m_max"][i]
            tmin    = daily["temperature_2m_min"][i]
            lluvia  = daily["precipitation_sum"][i]
            viento  = daily["wind_speed_10m_max"][i]
            codigo  = daily["weathercode"][i]
            condicion = "Despejado" if codigo < 3 else "Nublado" if codigo < 50 else "Lluvia"
            
            resultado += (
                f"\n{fecha}: {tmin}°C - {tmax}°C | "
                f"Viento: {viento} km/h | "
                f"Lluvia: {lluvia} mm | {condicion}"
            )
        return resultado
    except Exception as e:
        return f"Error al obtener pronóstico: {e}"


@tool
def evaluar_operacion(centro: str, operacion: str) -> str:
    """Evalúa si las condiciones climáticas son seguras para realizar una operación en Camanchaca.
    centro: ensenada, puelche o huito.
    operacion: cosecha, biometría o tratamiento."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado."
    
    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&current=temperature_2m,wind_speed_10m,precipitation,weathercode"
        f"&timezone=America/Santiago"
    )
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        current = data["current"]
        
        viento = current["wind_speed_10m"]
        lluvia = current["precipitation"]
        temp   = current["temperature_2m"]
        
        alertas = []
        if viento > 40:
            alertas.append(f" Viento peligroso: {viento} km/h (límite: 40 km/h)")
        if lluvia > 10:
            alertas.append(f" Lluvia intensa: {lluvia} mm")
        if temp < 5:
            alertas.append(f" Temperatura muy baja: {temp}°C")
        if temp > 18:
            alertas.append(f" Temperatura elevada: {temp}°C (riesgo para FCR)")
        
        if not alertas:
            return f" Condiciones APTAS para {operacion} en {datos['nombre']}."
        else:
            return f" Condiciones NO APTAS para {operacion} en {datos['nombre']}:\n" + "\n".join(alertas)
    except Exception as e:
        return f"Error al evaluar condiciones: {e}"


tools = [get_clima_actual, get_pronostico_semana, evaluar_operacion]

print("✓ Herramientas climáticas definidas.")
print(f"  Herramientas: {[t.name for t in tools]}")

✓ Herramientas climáticas definidas.
  Herramientas: ['get_clima_actual', 'get_pronostico_semana', 'evaluar_operacion']


In [4]:
from langgraph.prebuilt import create_react_agent as lg_create_react_agent

agent_executor = lg_create_react_agent(llm, tools)

print("✓ Agente y AgentExecutor listos.")

✓ Agente y AgentExecutor listos.


C:\Users\lenov\AppData\Local\Temp\ipykernel_13876\2672463633.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = lg_create_react_agent(llm, tools)


In [5]:
query = "¿Cuál es el clima actual en el centro Ensenada y es seguro hacer cosecha hoy?"

response = agent_executor.invoke({
    "messages": [{"role": "user", "content": query}]
})

print(f"\n🏁 Respuesta Final: {response['messages'][-1].content}")

El clima actual en el centro Ensenada (Piscicultura Petrohué) es el siguiente:
- **Temperatura:** 9.2°C
- **Viento:** 3.4 km/h
- **Precipitación:** 0.8 mm
- **Condición:** Lluvia

Además, las condiciones son **APTAS** para realizar la cosecha hoy.
🏁 Respuesta Final: El clima actual en el centro Ensenada (Piscicultura Petrohué) es el siguiente:
- **Temperatura:** 9.2°C
- **Viento:** 3.4 km/h
- **Precipitación:** 0.8 mm
- **Condición:** Lluvia

Además, las condiciones son **APTAS** para realizar la cosecha hoy.


In [6]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

print("=== MEMORIA MANUAL - CONVERSACIÓN CON EL OPERADOR ===\n")

print("1. Primera pregunta:")
query1 = "¿Cuál es el clima actual en el centro Ensenada?"

response1 = agent_executor.invoke({
    "messages": [{"role": "user", "content": query1}]
})
output1 = response1["messages"][-1].content
print(f"\nRespuesta: {output1}\n")

chat_history.append(HumanMessage(content=query1))
chat_history.append(AIMessage(content=output1))
print("Historial actualizado.\n")

print("2. Segunda pregunta (seguimiento):")
query2 = "¿Y es seguro hacer la cosecha allí hoy?"

messages_con_historia = (
    [{"role": "user" if isinstance(m, HumanMessage) else "assistant", "content": m.content}
     for m in chat_history]
    + [{"role": "user", "content": query2}]
)

response2 = agent_executor.invoke({"messages": messages_con_historia})
output2 = response2["messages"][-1].content
print(f"\nRespuesta: {output2}\n")

chat_history.append(HumanMessage(content=query2))
chat_history.append(AIMessage(content=output2))
print("Historial actualizado.\n")

print("3. Tercera pregunta (seguimiento):")
query3 = "¿De qué centro me estabas hablando?"

messages_con_historia = (
    [{"role": "user" if isinstance(m, HumanMessage) else "assistant", "content": m.content}
     for m in chat_history]
    + [{"role": "user", "content": query3}]
)

response3 = agent_executor.invoke({"messages": messages_con_historia})
output3 = response3["messages"][-1].content
print(f"\nRespuesta: {output3}\n")

print("=== CONTENIDO DE LA MEMORIA ===")
for msg in chat_history:
    tipo = "👤 Operador" if isinstance(msg, HumanMessage) else "🤖 Agente"
    print(f"{tipo}: {msg.content[:80]}...")

=== MEMORIA MANUAL - CONVERSACIÓN CON EL OPERADOR ===

1. Primera pregunta:
El clima actual en el centro Ensenada (Piscicultura Petrohué) es el siguiente:

- **Temperatura:** 9.2°C
- **Viento:** 3.4 km/h
- **Precipitación:** 0.8 mm
- **Condición:** Lluvia
Respuesta: El clima actual en el centro Ensenada (Piscicultura Petrohué) es el siguiente:

- **Temperatura:** 9.2°C
- **Viento:** 3.4 km/h
- **Precipitación:** 0.8 mm
- **Condición:** Lluvia

Historial actualizado.

2. Segunda pregunta (seguimiento):
Sí, las condiciones climáticas son aptas para realizar la cosecha hoy en el centro Ensenada (Piscicultura Petrohué).
Respuesta: Sí, las condiciones climáticas son aptas para realizar la cosecha hoy en el centro Ensenada (Piscicultura Petrohué).

Historial actualizado.

3. Tercera pregunta (seguimiento):
Te estaba hablando del centro **Ensenada**, también conocido como **Piscicultura Petrohué**. Este es el centro que mencionaste en tu consulta inicial.
Respuesta: Te estaba hablando del cen

In [7]:
# ============================================================
# SECCIÓN 5: CONVERSATION BUFFER MEMORY
# Implementación manual compatible con LangChain 1.3.1
# ============================================================

from langchain_core.messages import HumanMessage, AIMessage

class ConversationBufferMemory:
    """Buffer Memory manual compatible con LangChain 1.3.1"""
    def __init__(self):
        self.messages = []

    def load_history(self):
        return [
            {"role": "user" if isinstance(m, HumanMessage) else "assistant",
             "content": m.content}
            for m in self.messages
        ]

    def save_context(self, query, output):
        self.messages.append(HumanMessage(content=query))
        self.messages.append(AIMessage(content=output))

    def clear(self):
        self.messages = []


memory_buffer = ConversationBufferMemory()

def chat_buffer(query: str):
    history  = memory_buffer.load_history()
    messages = history + [{"role": "user", "content": query}]
    response = agent_executor.invoke({"messages": messages})
    output   = response["messages"][-1].content
    memory_buffer.save_context(query, output)
    return output

print("=== CONVERSATION BUFFER MEMORY ===")
print("Mantiene el historial completo de la conversación\n")

print("1. Primera pregunta:")
r1 = chat_buffer("¿Cómo está el clima en Puelche ahora?")
print(f"Respuesta: {r1}\n")

print("2. Segunda pregunta (seguimiento):")
r2 = chat_buffer("¿Hay riesgo de que afecte la biometría programada?")
print(f"Respuesta: {r2}\n")

print("=== ESTADO DE LA MEMORIA ===")
print(f"Total de mensajes almacenados: {len(memory_buffer.messages)}")
for i, msg in enumerate(memory_buffer.messages, 1):
    tipo = "👤 Operador" if isinstance(msg, HumanMessage) else "🤖 Agente"
    print(f"{i}. {tipo}: {msg.content[:60]}...")

=== CONVERSATION BUFFER MEMORY ===
Mantiene el historial completo de la conversación

1. Primera pregunta:
Actualmente, en el centro Puelche, el clima es el siguiente:

- **Temperatura:** 10.8°C
- **Viento:** 29.6 km/h
- **Precipitación:** 0.3 mm
- **Condición:** LluviaRespuesta: Actualmente, en el centro Puelche, el clima es el siguiente:

- **Temperatura:** 10.8°C
- **Viento:** 29.6 km/h
- **Precipitación:** 0.3 mm
- **Condición:** Lluvia

2. Segunda pregunta (seguimiento):
Las condiciones climáticas actuales en el centro Puelche son aptas para realizar la biometría programada. No hay riesgo que afecte la operación.Respuesta: Las condiciones climáticas actuales en el centro Puelche son aptas para realizar la biometría programada. No hay riesgo que afecte la operación.

=== ESTADO DE LA MEMORIA ===
Total de mensajes almacenados: 4
1. 👤 Operador: ¿Cómo está el clima en Puelche ahora?...
2. 🤖 Agente: Actualmente, en el centro Puelche, el clima es el siguiente:...
3. 👤 Operador: ¿Hay rie

In [8]:
# ============================================================
# SECCIÓN 6: CONVERSATION BUFFER WINDOW MEMORY (k=2)
# Solo recuerda los últimos k intercambios
# ============================================================

class ConversationWindowMemory:
    """Window Memory manual compatible con LangChain 1.3.1"""
    def __init__(self, k=2):
        self.k        = k
        self._all     = []

    def load_history(self):
        ventana = self._all[-(self.k * 2):]
        return [
            {"role": "user" if isinstance(m, HumanMessage) else "assistant",
             "content": m.content}
            for m in ventana
        ]

    def save_context(self, query, output):
        self._all.append(HumanMessage(content=query))
        self._all.append(AIMessage(content=output))

    @property
    def total(self):
        return len(self._all)

    @property
    def visible(self):
        return min(len(self._all), self.k * 2)


memory_window = ConversationWindowMemory(k=2)

def chat_window(query: str):
    history  = memory_window.load_history()
    messages = history + [{"role": "user", "content": query}]
    response = agent_executor.invoke({"messages": messages})
    output   = response["messages"][-1].content
    memory_window.save_context(query, output)
    return output

print("=== CONVERSATION BUFFER WINDOW MEMORY (k=2) ===")
print("Solo recuerda los últimos 2 intercambios\n")

interacciones = [
    "¿Cuál es el clima en Ensenada?",
    "¿Y en Puelche?",
    "¿Y en Huito?",
    "¿De qué centro me hablaste primero?"
]

for i, query in enumerate(interacciones, 1):
    print(f"{'='*20} INTERACCIÓN {i} {'='*20}")
    print(f"👤 Operador: {query}")
    respuesta = chat_window(query)
    print(f"\n📊 ESTADO DE LA MEMORIA:")
    print(f"   Total almacenado: {memory_window.total} mensajes")
    print(f"   Visible al modelo: {memory_window.visible} mensajes")
    print(f"   Descartados: {memory_window.total - memory_window.visible} mensajes\n")

=== CONVERSATION BUFFER WINDOW MEMORY (k=2) ===
Solo recuerda los últimos 2 intercambios

==================== INTERACCIÓN 1 ====================
👤 Operador: ¿Cuál es el clima en Ensenada?
El clima en Ensenada (Piscicultura Petrohué) actualmente es el siguiente:

- **Temperatura:** 9.2°C
- **Viento:** 3.4 km/h
- **Precipitación:** 0.8 mm
- **Condición:** Lluvia
📊 ESTADO DE LA MEMORIA:
   Total almacenado: 2 mensajes
   Visible al modelo: 2 mensajes
   Descartados: 0 mensajes

==================== INTERACCIÓN 2 ====================
👤 Operador: ¿Y en Puelche?
El clima en Puelche actualmente es el siguiente:

- **Temperatura:** 10.8°C
- **Viento:** 29.6 km/h
- **Precipitación:** 0.3 mm
- **Condición:** Lluvia
📊 ESTADO DE LA MEMORIA:
   Total almacenado: 4 mensajes
   Visible al modelo: 4 mensajes
   Descartados: 0 mensajes

==================== INTERACCIÓN 3 ====================
👤 Operador: ¿Y en Huito?
El clima en Huito (San José) actualmente es el siguiente:

- **Temperatura:** 10.6°C
-

In [9]:
# ============================================================
# SECCIÓN 7: CONVERSATION SUMMARY MEMORY
# Resume la conversación para ahorrar tokens
# ============================================================

class ConversationSummaryMemory:
    """Summary Memory manual compatible con LangChain 1.3.1"""
    def __init__(self, llm, max_messages=6):
        self.llm          = llm
        self.max_messages = max_messages
        self.messages     = []
        self.summary      = None

    def _resumir(self):
        if len(self.messages) <= self.max_messages:
            return
        to_summarize = self.messages[:-2]
        text = "\n".join([
            f"{'Usuario' if isinstance(m, HumanMessage) else 'Asistente'}: {m.content}"
            for m in to_summarize
        ])
        res = self.llm.invoke(
            f"Resume en 2-3 líneas esta conversación:\n{text}"
        )
        self.summary  = res.content
        self.messages = self.messages[-2:]

    def load_history(self):
        history = []
        if self.summary:
            history.append({
                "role":    "assistant",
                "content": f"Resumen de conversación anterior: {self.summary}"
            })
        history += [
            {"role": "user" if isinstance(m, HumanMessage) else "assistant",
             "content": m.content}
            for m in self.messages
        ]
        return history

    def save_context(self, query, output):
        self.messages.append(HumanMessage(content=query))
        self.messages.append(AIMessage(content=output))
        self._resumir()


memory_summary = ConversationSummaryMemory(llm=llm)

def chat_summary(query: str):
    history  = memory_summary.load_history()
    messages = history + [{"role": "user", "content": query}]
    response = agent_executor.invoke({"messages": messages})
    output   = response["messages"][-1].content
    memory_summary.save_context(query, output)
    return output

print("=== CONVERSATION SUMMARY MEMORY ===")
print("Resume conversaciones largas para ahorrar tokens\n")

consultas = [
    "Soy Carlos, jefe del centro Ensenada. ¿Cómo está el clima hoy?",
    "¿Es seguro programar la cosecha para mañana?",
    "¿Y qué hay del pronóstico para el resto de la semana?",
    "¿Sobre qué estábamos hablando y qué decisiones tomamos?"
]

for i, query in enumerate(consultas, 1):
    print(f"{'='*15} INTERACCIÓN {i} {'='*15}")
    print(f"👤 Operador: {query}")
    respuesta = chat_summary(query)
    print(f"\n📊 ESTADO DE LA MEMORIA:")
    print(f"   Mensajes en contexto: {len(memory_summary.messages)}")
    print(f"   Tiene resumen: {'✅ Sí' if memory_summary.summary else '❌ No'}\n")

print("\n=== COMPARACIÓN DE ESTRATEGIAS ===")
print("Buffer Memory:        Historial completo. Ideal para conversaciones cortas.")
print("Window Memory (k=2):  Solo últimos 2 intercambios. Balance costo/contexto.")
print("Summary Memory:       Resume el historial. Ideal para conversaciones largas.")

=== CONVERSATION SUMMARY MEMORY ===
Resume conversaciones largas para ahorrar tokens

=============== INTERACCIÓN 1 ===============
👤 Operador: Soy Carlos, jefe del centro Ensenada. ¿Cómo está el clima hoy?
Hola Carlos, el clima en el centro Ensenada hoy es el siguiente:

- **Temperatura:** 9.2°C
- **Viento:** 3.4 km/h
- **Precipitación:** 0.8 mm
- **Condición:** Lluvia

Si necesitas más información o evaluar alguna operación, no dudes en pedírmelo.
📊 ESTADO DE LA MEMORIA:
   Mensajes en contexto: 2
   Tiene resumen: ❌ No

=============== INTERACCIÓN 2 ===============
👤 Operador: ¿Es seguro programar la cosecha para mañana?
Sí, las condiciones climáticas son aptas para programar la cosecha en el centro Ensenada mañana. Puedes proceder con la planificación. Si necesitas más detalles, estoy aquí para ayudarte.
📊 ESTADO DE LA MEMORIA:
   Mensajes en contexto: 4
   Tiene resumen: ❌ No

=============== INTERACCIÓN 3 ===============
👤 Operador: ¿Y qué hay del pronóstico para el resto de la s

In [10]:
# ============================================================
# SECCIÓN 8: DEMO CONVERSACIÓN COMPLETA CON MEMORIA
# Simula una jornada operativa real usando InMemoryChatMessageHistory
# ============================================================

from langchain_core.chat_history import InMemoryChatMessageHistory

history_store = {}

def get_session_history(session_id: str):
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

def invocar_con_sesion(query: str, session_id: str):
    hist     = get_session_history(session_id)
    messages = [
        {"role": "user" if isinstance(m, HumanMessage) else "assistant",
         "content": m.content}
        for m in hist.messages
    ] + [{"role": "user", "content": query}]

    response = agent_executor.invoke({"messages": messages})
    output   = response["messages"][-1].content

    hist.add_user_message(query)
    hist.add_ai_message(output)
    return output

print("=== SIMULACIÓN JORNADA OPERATIVA CAMANCHACA ===\n")

session_id = "turno_manana_001"

jornada = [
    ("Carlos - Jefe Ensenada",  "Buenos días. ¿Cómo están las condiciones en Ensenada para hoy?"),
    ("Carlos - Jefe Ensenada",  "¿Podemos proceder con la cosecha programada para esta mañana?"),
    ("María - Jefe Puelche",    "Hola, me acabo de unir. ¿Qué tal el clima en Puelche esta semana?"),
    ("Carlos - Jefe Ensenada",  "¿Recuerdas qué operación tenía programada yo esta mañana?"),
]

for operador, consulta in jornada:
    print(f"👤 {operador}: {consulta}")
    respuesta = invocar_con_sesion(consulta, session_id)
    print(f"🤖 Agente: {respuesta}\n")

print("=== HISTORIAL DE LA SESIÓN ===")
historial = history_store[session_id].messages
print(f"Total de mensajes: {len(historial)}")
for i, msg in enumerate(historial, 1):
    rol = "👤 Operador" if msg.type == "human" else "🤖 Agente"
    print(f"{i}. {rol}: {msg.content[:70]}...")

=== SIMULACIÓN JORNADA OPERATIVA CAMANCHACA ===

👤 Carlos - Jefe Ensenada: Buenos días. ¿Cómo están las condiciones en Ensenada para hoy?
Buenos días. Las condiciones actuales en Ensenada (Piscicultura Petrohué) son las siguientes:

- **Temperatura:** 9.2°C
- **Viento:** 3.4 km/h
- **Precipitación:** 0.8 mm
- **Condición:** Lluvia

Si necesitas más información o evaluar alguna operación, no dudes en pedírmelo.🤖 Agente: Buenos días. Las condiciones actuales en Ensenada (Piscicultura Petrohué) son las siguientes:

- **Temperatura:** 9.2°C
- **Viento:** 3.4 km/h
- **Precipitación:** 0.8 mm
- **Condición:** Lluvia

Si necesitas más información o evaluar alguna operación, no dudes en pedírmelo.

👤 Carlos - Jefe Ensenada: ¿Podemos proceder con la cosecha programada para esta mañana?
Sí, las condiciones climáticas son aptas para proceder con la cosecha programada esta mañana en Ensenada (Piscicultura Petrohué). ¡Éxito en la operación!🤖 Agente: Sí, las condiciones climáticas son aptas para pro